In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from rl_setup import simulation_parameters, STEM_environment_history, CNNModelWithHistory
import os
import ray
from ray.tune.registry import register_env
from ray.rllib.algorithms.sac import SAC, SACConfig
from ray.rllib.algorithms.ppo import PPO, PPOConfig
from tqdm import tqdm
from ray.rllib.models import ModelCatalog

# GPU configuration
GPU_USE = 0
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_USE)
os.environ["OMP_NUM_THREADS"] = "1"

2025-11-12 17:54:22.958858: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-12 17:54:22.968487: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762998862.981414  530076 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762998862.985685  530076 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762998863.000044  530076 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [4]:
ModelCatalog.register_custom_model("cnn_with_history", CNNModelWithHistory)
print("Custom model registered successfully!")

Custom model registered successfully!


In [7]:
# Initialize Ray
ray.init(ignore_reinit_error=True,
         include_dashboard=False,
         num_cpus=2,
         num_gpus=1)

# Register the custom environment
def env_creator(cfg):
    return STEM_environment_history(config=cfg)

register_env("stem_env_history", env_creator)

# PPO Configuration with custom model
config = PPOConfig()

# Disable new API stack to use custom_model
config.api_stack(
    enable_rl_module_and_learner=False,
    enable_env_runner_and_connector_v2=False,
)

# Create a temporary environment to get spaces
temp_env = STEM_environment_history(config=simulation_parameters)
obs_space = temp_env.observation_space
print(type(obs_space))
action_space = temp_env.action_space
temp_env.close()

# Environment configuration with explicit spaces
config.environment(
    env="stem_env_history",
    env_config=simulation_parameters,
    observation_space=obs_space,
    action_space=action_space,
)

# Framework configuration
config.framework(framework="torch")

# Resource configuration
config.resources(num_gpus=1)

# Training configuration
config.training(
    # Use custom model
    model={
        "custom_model": "cnn_with_history",
        "custom_model_config": {},
        "vf_share_layers": False,  # Separate value function
    },
    # PPO hyperparameters
    train_batch_size=1000,
    num_sgd_iter=10,
    
    lr=3e-4,
    lr_schedule=None,
    
    gamma=0.99,
    lambda_=0.95,  # GAE lambda
    
    clip_param=0.2,
    vf_clip_param=10.0,
    
    entropy_coeff=0.01,
    entropy_coeff_schedule=[
        [0, 0.01],
        [200, 0.001],
    ],
    vf_loss_coeff=0.5,
    grad_clip=0.5,
    kl_coeff=0.2,
    kl_target=0.01,
)

# Build the PPO trainer
trainer = config.build()

print("PPO Trainer with custom CNN+history model initialized successfully!")

2025-11-12 17:56:48,010	INFO worker.py:1850 -- Calling ray.init() again after it has already been called.
2025-11-12 17:56:48,013	WARNING deprecation.py:50 -- DeprecationWarning: `config.training(num_sgd_iter=..)` has been deprecated. Use `config.training(num_epochs=..)` instead. This will raise an error in the future!


<class 'gymnasium.spaces.dict.Dict'>


(pid=532358) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=532358) E0000 00:00:1762999008.628070  532358 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=532358) E0000 00:00:1762999008.634364  532358 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
(pid=532358) W0000 00:00:1762999008.648684  532358 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=532358) W0000 00:00:1762999008.648708  532358 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=532358) W0000 00:00:1762999008.648711  532358 computation_placer.cc:177] computation placer already registered. Please check linkage and avo

(RolloutWorker pid=532358) <class 'gymnasium.spaces.box.Box'>
<class 'gymnasium.spaces.box.Box'>


AttributeError: 'Box' object has no attribute 'spaces'

In [6]:
# Initialize Ray
ray.init(ignore_reinit_error=True,
         include_dashboard=False,
         num_cpus=2,
         num_gpus=1)

# Register the custom environment
def env_creator(cfg):
    return STEM_environment_history(config=cfg)

register_env("stem_env_history", env_creator)

# PPO Configuration with custom model
config = PPOConfig()

# Disable new API stack to use custom_model
config.api_stack(
    enable_rl_module_and_learner=False,
    enable_env_runner_and_connector_v2=False,
)

# Create a temporary environment to get spaces
temp_env = STEM_environment_history(config=simulation_parameters)
obs_space = temp_env.observation_space
action_space = temp_env.action_space

# Debug: Check what the observation space actually is
print(f"Observation space type: {type(obs_space)}")
print(f"Observation space: {obs_space}")
if hasattr(obs_space, 'spaces'):
    print(f"Observation space keys: {obs_space.spaces.keys()}")
    for key, space in obs_space.spaces.items():
        print(f"  {key}: {space}")

temp_env.close()

# Environment configuration with explicit spaces
config.environment(
    env="stem_env_history",
    env_config=simulation_parameters,
    observation_space=obs_space,
    action_space=action_space,
)

# Framework configuration
config.framework(framework="torch")

# Rollout configuration - IMPORTANT: Add this
config.rollouts(
    num_rollout_workers=0,
    create_env_on_local_worker=True,
    # Disable preprocessing that might flatten Dict spaces
    preprocessor_pref=None,
)

# Resource configuration
config.resources(num_gpus=1)

# Training configuration
config.training(
    # Use custom model
    model={
        "custom_model": "cnn_with_history",
        "custom_model_config": {},
        "vf_share_layers": False,
        # Disable default preprocessors
        "_disable_preprocessor_api": True,
    },
    # PPO hyperparameters
    train_batch_size=1000,
    num_sgd_iter=10,
    
    lr=3e-4,
    lr_schedule=None,
    
    gamma=0.99,
    lambda_=0.95,
    
    clip_param=0.2,
    vf_clip_param=10.0,
    
    entropy_coeff=0.01,
    entropy_coeff_schedule=[
        [0, 0.01],
        [200, 0.001],
    ],
    vf_loss_coeff=0.5,
    grad_clip=0.5,
    kl_coeff=0.2,
    kl_target=0.01,
)

# Build the PPO trainer
trainer = config.build()

print("PPO Trainer with custom CNN+history model initialized successfully!")

2025-11-12 17:51:57,633	INFO worker.py:1850 -- Calling ray.init() again after it has already been called.
2025-11-12 17:51:57,636	WARNING deprecation.py:50 -- DeprecationWarning: `config.training(num_sgd_iter=..)` has been deprecated. Use `config.training(num_epochs=..)` instead. This will raise an error in the future!


Observation space type: <class 'gymnasium.spaces.dict.Dict'>
Observation space: Dict('actions': Box(-1.0, 1.0, (2,), float32), 'images': Box(0.0, 1.0, (2, 64, 64), float32))
Observation space keys: dict_keys(['actions', 'images'])
  actions: Box(-1.0, 1.0, (2,), float32)
  images: Box(0.0, 1.0, (2, 64, 64), float32)


(pid=528066) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=528066) E0000 00:00:1762998718.225152  528066 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=528066) E0000 00:00:1762998718.229508  528066 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
(pid=528066) W0000 00:00:1762998718.239616  528066 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=528066) W0000 00:00:1762998718.239640  528066 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
(pid=528066) W0000 00:00:1762998718.239643  528066 computation_placer.cc:177] computation placer already registered. Please check linkage and avo

TypeError: 'Box' object is not subscriptable